In [1]:
import os
from collections import defaultdict
from datetime import datetime
from functools import partial
from pathlib import Path

import lightning as pl
import stable_pretraining as spt  # also registers the ${eval:...} resolver used in data/*.yaml
import stable_worldmodel as swm
import torch
from lightning.pytorch.loggers import WandbLogger
from omegaconf import OmegaConf, open_dict

from jepa import JEPA
from module import ARPredictor, Embedder, MLP, SIGReg

from utils import get_column_normalizer, get_img_preprocessor, ModelObjectCallBack

In [2]:
# CONFIG_DIR = Path(__file__).parent / "config" / "train"
CONFIG_DIR = Path(os.path.abspath('./config/train/'))
CONFIG_DIR

PosixPath('/home/yvasiliuk/Fast-LeWorldModel/config/train')

In [3]:
def load_cfg(data: str = "pusht"):
    cfg = OmegaConf.load(CONFIG_DIR / "Fast-lewm.yaml")
    cfg.pop("defaults")
    cfg.data = OmegaConf.load(CONFIG_DIR / "data" / f"{data}.yaml")
    cfg.subdir = datetime.now().strftime("%Y%m%d-%H%M%S")  # replaces ${hydra:job.id}

    # --- knobs ---------------------------------------------------------------
    # RAM in flight ~= num_workers * prefetch_factor * batch_size
    #                  * (max_transitions * action_frame_skip + 1) frames * 602 KB
    cfg.loader.batch_size = 16
    cfg.num_workers = 6
    cfg.loader.prefetch_factor = 3
    cfg.wm.action_frame_skip = 5
    cfg.wm.dense_supervision.max_transitions = 5
    cfg.wandb.enabled = True
    return cfg

In [4]:
cfg = load_cfg("pusht")

In [5]:
run_dir = Path("results") / f"plain2-{cfg.subdir}"
run_dir.mkdir(parents=True, exist_ok=True)

In [6]:
cfg.seed

3072

In [7]:
dense_cfg = cfg.wm.get("dense_supervision", None)
dense_cfg

{'enabled': True, 'supervise_last_state_only': False, 'transition_sampling_strategy': 'max_adaptive', 'min_transitions': 1, 'max_transitions': 5}

In [8]:
action_frame_skip = int(cfg.wm.get("action_frame_skip", 1))
action_frame_skip

5

In [9]:
base_dataset_kwargs = OmegaConf.to_container(cfg.data.dataset, resolve=True)
base_dataset_kwargs

{'num_steps': 6,
 'frameskip': 1,
 'name': 'pusht_expert_train',
 'keys_to_load': ['pixels', 'action', 'proprio', 'state'],
 'keys_to_cache': ['action', 'proprio', 'state']}

In [10]:
configured_num_steps = int(base_dataset_kwargs["num_steps"])
configured_num_steps

6

In [11]:
base_dataset_kwargs["num_steps"] = ((configured_num_steps - 1) * action_frame_skip + 1)
base_dataset_kwargs['num_steps']

26

In [12]:
# if "keys_to_cache" in base_dataset_kwargs:
base_dataset_kwargs["keys_to_cache"] = []

In [13]:
base_dataset_kwargs['cache_dir'] = os.path.expanduser('~/stablewm_data/')
base_dataset_kwargs['cache_dir']

'/home/yvasiliuk/stablewm_data/'

In [14]:
base_dataset = swm.data.HDF5Dataset(**base_dataset_kwargs, transform=None)

In [15]:
transforms = [get_img_preprocessor(source="pixels", target="pixels", img_size=cfg.img_size)]
transforms

[Compose(
   args=(ToImage(
     source=pixels, target=pixels
     (t): Compose(
           <function to_image at 0x7e1e903723b0>
           ToDtype(scale=True)
           Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
     )
   ), Resize(size=[224], interpolation=2, antialias=True, source=pixels, target=pixels))
 )]

In [16]:
cfg.data.dataset.keys_to_load

['pixels', 'action', 'proprio', 'state']

In [17]:
with open_dict(cfg):
    for col in cfg.data.dataset.keys_to_load:
        if col.startswith("pixels"):
            continue

        normalizer = get_column_normalizer(base_dataset, col, col)
        transforms.append(normalizer)
        setattr(cfg.wm, f"{col}_dim", base_dataset.get_dim(col))

    if hasattr(cfg.wm, "action_dim"):
        base_action_dim = int(cfg.wm.action_dim)
        cfg.wm.action_base_dim = base_action_dim
        cfg.wm.action_dim = int(base_action_dim * action_frame_skip)
    cfg.wm.action_frame_skip = int(action_frame_skip)


In [18]:
transform = spt.data.transforms.Compose(*transforms)
base_dataset.transform = transform
transform

Compose(
  args=(Compose(
    args=(ToImage(
      source=pixels, target=pixels
      (t): Compose(
            <function to_image at 0x7e1e903723b0>
            ToDtype(scale=True)
            Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
      )
    ), Resize(size=[224], interpolation=2, antialias=True, source=pixels, target=pixels))
  ), WrapTorchTransform(norm_fn, types=['object']), WrapTorchTransform(norm_fn, types=['object']), WrapTorchTransform(norm_fn, types=['object']))
)

In [19]:
rnd_gen = torch.Generator().manual_seed(int(cfg.seed))

In [20]:
min_transitions = int(dense_cfg.get("min_transitions", 1))
max_transitions = int(dense_cfg.get("max_transitions", min_transitions))
min_transitions, max_transitions

(1, 5)

In [22]:
type(cfg.data.dataset)

omegaconf.dictconfig.DictConfig

In [23]:
dataset_kwargs = OmegaConf.to_container(cfg.data.dataset, resolve=True)
dataset_kwargs

{'num_steps': 6,
 'frameskip': 1,
 'name': 'pusht_expert_train',
 'keys_to_load': ['pixels', 'action', 'proprio', 'state'],
 'keys_to_cache': ['action', 'proprio', 'state']}

In [24]:
reference_transitions = int(min_transitions)
reference_transitions


1

In [26]:
aux_keys_to_cache_cfg = dense_cfg.get("aux_keys_to_cache", [])
aux_keys_to_cache_cfg

[]

In [27]:
aux_keys_to_cache = (None if aux_keys_to_cache_cfg is None else list(aux_keys_to_cache_cfg))
aux_keys_to_cache

[]

In [28]:
requested_transitions = list(range(min_transitions, max_transitions + 1))
datasets_by_transitions = {}
requested_transitions

[1, 2, 3, 4, 5]

In [29]:
reference_transitions

1

In [30]:
for transitions in requested_transitions:
    cur_kwargs = dict(dataset_kwargs)
    cur_kwargs["num_steps"] = int(transitions * action_frame_skip + 1)
    if (
        transitions != reference_transitions
        and aux_keys_to_cache is not None
        and "keys_to_cache" in cur_kwargs
    ):
        cur_kwargs["keys_to_cache"] = aux_keys_to_cache
    datasets_by_transitions[transitions] = swm.data.HDF5Dataset(**cur_kwargs, transform=transform)

15:28:50.469 | INFO    (8332, stable_pretraining) | Cached 'action' from '/home/yvasiliuk/.stable_worldmodel/pusht_expert_train.h5'
15:28:50.499 | INFO    (8332, stable_pretraining) | Cached 'proprio' from '/home/yvasiliuk/.stable_worldmodel/pusht_expert_train.h5'
15:28:50.540 | INFO    (8332, stable_pretraining) | Cached 'state' from '/home/yvasiliuk/.stable_worldmodel/pusht_expert_train.h5'


In [31]:
datasets_by_transitions

{1: <stable_worldmodel.data.dataset.HDF5Dataset at 0x7e1e80f203a0>,
 2: <stable_worldmodel.data.dataset.HDF5Dataset at 0x7e1e82a4ad40>,
 3: <stable_worldmodel.data.dataset.HDF5Dataset at 0x7e1e82a4abc0>,
 4: <stable_worldmodel.data.dataset.HDF5Dataset at 0x7e1e8a0b65f0>,
 5: <stable_worldmodel.data.dataset.HDF5Dataset at 0x7e1e80313fd0>}

In [32]:
reference_dataset = datasets_by_transitions[reference_transitions]
reference_dataset

In [33]:
train_subset, _ = spt.data.random_split(
        reference_dataset,
        lengths=[cfg.train_split, 1 - cfg.train_split],
        generator=rnd_gen,
    )

In [34]:
train_subset

In [35]:
val_source_dataset = datasets_by_transitions[int(max_transitions)]
val_gen = torch.Generator().manual_seed(int(cfg.seed) + 1)

In [36]:
_, val_set = spt.data.random_split(
        val_source_dataset,
        lengths=[cfg.train_split, 1 - cfg.train_split],
        generator=val_gen,
    )

In [40]:
from train import _resolve_alignment_mode, _episode_column_name

In [41]:
base_mode = _resolve_alignment_mode(reference_dataset)
base_mode

'global_index'

In [42]:
episode_col = _episode_column_name(reference_dataset)
episode_col

''

In [44]:
base_keys = [(0, int(i)) for i in train_subset.indices]
base_keys.__len__()

2018980

In [46]:
from train import FixedStartVariableTransitionsDataset

In [47]:
train_set = FixedStartVariableTransitionsDataset(
        datasets_by_transitions=datasets_by_transitions,
        base_keys=base_keys,
        fallback_transitions=reference_transitions,
    )

In [48]:
len(train_set)

2018980

In [51]:
# sample = train_set.__getitem__(base_keys[0])

In [52]:
resolved_transitions_by_local = [train_set.resolve_local_transitions(i, max_transitions) for i in range(len(train_set))]
len(resolved_transitions_by_local)

2018980

In [55]:
resolution_hist = defaultdict(int)
for transitions in resolved_transitions_by_local:
    resolution_hist[int(transitions)] += 1

hist_summary = ", ".join(f"{t}:{resolution_hist[t]}" for t in sorted(resolution_hist.keys()))
print("Dense transition max-adaptive buckets (transitions:count): " f"{hist_summary}")

Dense transition max-adaptive buckets (transitions:count): 1:84035, 2:84036, 3:84112, 4:84058, 5:1682739


In [56]:
batch_size = int(cfg.loader.batch_size)
drop_last = bool(cfg.loader.get("drop_last", True))
drop_last

True

In [59]:
from train import FixedCoverageMaxTransitionsBatchSampler

In [60]:
train_batch_sampler = FixedCoverageMaxTransitionsBatchSampler(
    num_samples=len(train_set),
    batch_size=batch_size,
    resolved_transitions_by_local=resolved_transitions_by_local,
    drop_last=drop_last,
    seed=int(cfg.seed),
)

In [61]:
train_loader_kwargs = OmegaConf.to_container(cfg.loader, resolve=True)
train_loader_kwargs

{'batch_size': 16,
 'num_workers': 6,
 'persistent_workers': True,
 'prefetch_factor': 3,
 'pin_memory': True}

In [62]:
train_loader_kwargs.pop("batch_size", None)
train_loader_kwargs.pop("shuffle", None)
train_loader_kwargs.pop("drop_last", None)
train_loader_kwargs

{'num_workers': 6,
 'persistent_workers': True,
 'prefetch_factor': 3,
 'pin_memory': True}

In [63]:
train_loader = torch.utils.data.DataLoader(train_set,
    batch_sampler=train_batch_sampler,
    generator=rnd_gen,
    **train_loader_kwargs,
)

val_loader = torch.utils.data.DataLoader(val_set,
    **cfg.loader,
    shuffle=False,
    drop_last=False,
)

In [64]:
train_loader.batch_size

In [66]:
for step, batch in enumerate(train_loader):
    print(step)
    break

0


In [68]:
batch.keys()

dict_keys(['pixels', 'action', 'proprio', 'state', 'dense_transitions'])

In [70]:
batch['dense_transitions']

tensor([5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5])

In [71]:
batch['pixels'].shape

torch.Size([16, 26, 3, 224, 224])

In [73]:
batch['action'].shape

torch.Size([16, 26, 2])

In [75]:
batch['state'][0, 0]

tensor([-0.3956,  1.7401, -1.3114,  1.6474,  1.5642, -0.9432, -1.0083])